In [50]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader


# API call

In [51]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
print(api_key[:10] + "...")

AQ.Ab8RN6K...


In [52]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=api_key
)

response = llm.invoke("Hello")
print(response.content)

[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EvwFCvkFAQw51sekYUGLUbwSk0qFlnl9vubRev0WFV8ajV3rK2p3lZ8HbawTIH3VKtLHj79g34UnnAV4fDaJS6b7u22atfgCtUkeN2b+5mjSStDFrhe1WX8bnKKGMF58WOOYesfPqbeXmuaunerZkhYA6A3zbvdycSSYApvuuZxoUzk+Tz65Q4YTmESbFxVFtMcjiOJ8uOOTYnSo0u1N/8S9OBtrS+S4vZIOoyGHhkxpA5xqWYj0nnyXMy7kDl/540r/Ta4Q5fRKtn2he24qnsp4kKNQwk/drGhdK8SKHznZEKjJgVLWRYX2Bom2BFS9082HCVii7u584V5unSv/Sd++gH73GqcERr2R2Ru5PXqc2QyJvB0uPXZNoWXjqKjoeTSOcNUS3/rau23F0YowilUZUMEIeKEpILDe9p4Z9/nBWNsfYc7E0qFLMcb4mvqdRsHn2ZWlyJtdR6DI1tOpaXowhOIi39TSNT3DKpylu+cMXZz1P4sADK7iBkNKae84VxEz7fX2F4TSzx2U6mUs550YRw3+4xQqjXZcEGCtjkhrxeai44RZhdP8eIA+3MGrWAKazXfoWr2QflqLbQTMumNSJntAoNviPf13OU3vlQQ2iv6QLPuEPTJ29mxY0nRjaKSczdLJUbVhoYW3OBh0hmAWoLNf1Q8IlEuekc2mzQMlgArtIF0caomA6bmv4S//bzGiVF0mVqU+/GEoLNBlqy9cjRkDZxVZtR9mb5lomzaKBf1ZmTeZK5QnjenC+JLF+SGeU5+XksJLpzoFDVwASNbhqdkVN6G3uMgQYvCJjWa71zYGx1zskXJA3oFC6f7QBj2WGaTctBmGd93D983LeBVsTW0up2ch28IzK6TZ2uf3FaDkbv/0HFTNc3dO0fa14iKR2pXOcQaonznbL

## Knowledge Base import

In [53]:
loader = DirectoryLoader(
    "knowledge_base",
    glob="**/*.md",
    loader_cls=lambda path: TextLoader(path, encoding="utf-8")
)

documents = loader.load()
print(len(documents))

4


In [54]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

In [55]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10270.38it/s]


In [56]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [57]:
vectorstore.persist()

In [58]:
query = "Why is my LED not turning on?"

In [59]:
results = vectorstore.similarity_search(
    query,
    k=3
)

In [60]:
for i, doc in enumerate(results):
    print(f"\nRESULT {i+1}")
    print("-"*50)
    print(doc.page_content[:500])


RESULT 1
--------------------------------------------------
## TO TURN LED OFF

We can turn OFF two or more LED together by choosing pins from the board and set pinMode(pin number, OUTPUT) and digitalWrite (pin number, HIGH/LOW). Edit the pin number according to the pin where you put your led on the board (Figure 1.22).

## TO BLINK THE LED

We can make two or more LED lights blink together by choosing pins from the board, set pinMode(pin number, OUTPUT) and digitalWrite (pin number, HIGH/LOW) and by using delay () for one second delay use 1000 for two s

RESULT 2
--------------------------------------------------
## TO TURN LED OFF

We can turn OFF two or more LED together by choosing pins from the board and set pinMode(pin number, OUTPUT) and digitalWrite (pin number, HIGH/LOW). Edit the pin number according to the pin where you put your led on the board (Figure 1.22).

## TO BLINK THE LED

We can make two or more LED lights blink together by choosing pins from the board, set pinMod

In [61]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [62]:
query = "My LED is not turning on. What should I check?"

In [63]:
docs = retriever.invoke(query)

In [64]:
context = "\n\n".join(
    doc.page_content
    for doc in docs
)

In [65]:
prompt = f"""
You are a robotics troubleshooting assistant.

Use ONLY the provided context.

Context:
{context}

Question:
{query}
"""

In [69]:
response = llm.invoke(prompt)

In [70]:
print(response.content[0]["text"])

Based on the provided context, you should check the following to troubleshoot your LED:

1. **Check your code and pin configurations:**
   * Ensure you have set `pinMode(pin number, OUTPUT)` for the correct pin.
   * Check your `digitalWrite(pin number, HIGH/LOW)` settings.
   * Verify that the pin number in your code matches the physical pin on the board where you placed your LED.

2. **Check the physical connections:**
   * Verify that the LED is connected to **GND** and **5VF** (as shown in Figure 1.10).
